In [4]:
import requests
import urllib.parse
import json
import os
import time

def search_world_news(keywords, starting_date, ending_date):
    """
    Search articles using World News API with multiple keywords (ORed together).
    
    Parameters
    ----------
    keywords : list of str
        Keywords to search for. The search will match any of these (using OR).
    starting_date : str
        Earliest publish date, format YYYY-MM-DD.
    ending_date : str
        Latest publish date, format YYYY-MM-DD.
    language : str, optional
        Language code, default "en".
    """
    # api_key = "96c129136e4d47ae9f6234455a0841fd"
    api_key = "0d033f49793b4de5a06b3cf353d2ed80"
    base_url = "https://api.worldnewsapi.com/search-news"
    
    # Build the text parameter: join keywords with ' OR '
    text_query = " OR ".join(keywords)

    params = {
        "entities" : 'ORG:Suruhanjaya Syarikat Malaysia',
        # "text": 'Companies Commission of Malaysia OR Suruhanjaya Syarikat Malaysia',
        "earliest-publish-date": starting_date,
        "latest-publish-date": ending_date,
        "source-country": "my",
        "number": 10,
        "offset": 0,
        "api-key": api_key
    }
    
    all_news = []
    

    # Initial request to get the total number of available articles
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    initial_data = response.json()
    
    if initial_data and "available" in initial_data and "news" in initial_data:
        available_articles = initial_data["available"]
        news_per_page = initial_data["number"]
        # all_news.extend(initial_data["news"]) # why need to extend? it's empty
        all_news = initial_data['news']
        
        if available_articles > news_per_page:
            num_pages = (available_articles + news_per_page - 1) // news_per_page
            
            for page in range(2, num_pages + 1):
                params["offset"] = (page - 1) * news_per_page
                print(f"Fetching page {page} (offset: {params['offset']})...")
                response = requests.get(base_url, params=params)
                response.raise_for_status()
                data = response.json()
                if data and "news" in data:
                    all_news.extend(data["news"])
                else:
                    print(f"Error: No 'news' found in response for page {page}.")
                    break
                time.sleep(0.1)
        
        print(f"Successfully retrieved {len(all_news)} out of {available_articles} available articles.")
        
        # Process sentiment for each news item
        print("\nProcessing sentiment analysis:")
        for news_item in all_news:
            if 'sentiment' in news_item:
                news_item['sentiment_score'] = news_item.pop('sentiment')
            
            sentiment_score = news_item.get('sentiment_score', 0)
            if sentiment_score > 0.2:
                news_item['sentiment'] = 'Positive'
                print(f"[Positive] {news_item.get('title', 'No title')}")
            elif sentiment_score < -0.2:
                news_item['sentiment'] = 'Negative'
                print(f"[Negative] {news_item.get('title', 'No title')}")
            else:
                news_item['sentiment'] = 'Neutral'
                print(f"[Neutral] {news_item.get('title', 'No title')}")
        
        output_dir = "logs"
        filename = "news_results.json"
        filepath = os.path.join(output_dir, filename)
        output = {"total_available": available_articles, "news": all_news}
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=4, ensure_ascii=False)
        
        print(f"\nAll news saved to '{filepath}'")
        return output


In [5]:

result = search_world_news(
    keywords=["Suruhanjaya Syarikat Malaysia","Companies Commission of Malaysia"],
            #   , "SSM", "CCM"],
    starting_date="2025-10-05",
    ending_date="2025-10-14"
)


Successfully retrieved 0 out of 0 available articles.

Processing sentiment analysis:

All news saved to 'logs\news_results.json'


In [ ]:
Successfully retrieved 7 out of 7 available articles.

Processing sentiment analysis:
[Neutral] The hidden costs of Malaysia’s revenue leakage explained
[Neutral] Wagyu House director charged with falsifying company documents
[Neutral] Suhakam: Ulu Baram probe separate from ongoing court case
[Positive] Pakistan PM concludes three-day visit to Malaysia
[Negative] Court fixes Nov 5 to decide on judicial review bid by Daim’s family
[Neutral] 2026 budget: the key highlights
[Neutral] Budget 2026: RM3,000 childcare tax relief expanded to cover children up to 12 years old

In [4]:
result

{'total_available': 1,
 'news': [{'id': 367030368,
   'title': 'Court fixes Nov 5 to decide on judicial review bid by Daim’s family',
   'text': 'The applicants included Daim Zainuddin\'s wife Naimah Khalid and their four children, Asnida, Wira Dani, Amir Zainuddin and Amin Zainuddin. (Bernama pic)\nKUALA LUMPUR: The High Court has fixed Nov 5 to deliver its decision on an application by the family of the late Daim Zainuddin to obtain leave to initiate judicial review proceedings over the seizure and freezing of their bank accounts.\nJudge Aliza Sulaiman set the date after hearing submissions from senior federal counsel Nurhafizza Azizan, representing the attorney-general, while lawyer Gurdial Singh Nijar represented Daim and his family.\nEarlier, Gurdial submitted that the applicants had established that there was arguably a clear indication that the respondents had abused their powers by invoking different statutes to indefinitely prolong the freezing or seizure of the applicants\' p

In [8]:
keywords=["Suruhanjaya Syarikat Malaysia","Companies Commission of Malaysia"]
text_query = " OR ".join(keywords)
print(text_query)

Suruhanjaya Syarikat Malaysia OR Companies Commission of Malaysia


In [10]:
import worldnewsapi
from worldnewsapi.models.search_news200_response import SearchNews200Response
from worldnewsapi.rest import ApiException
from pprint import pprint

# Defining the host is optional and defaults to https://api.worldnewsapi.com
# See configuration.py for a list of all supported configuration parameters.
configuration = worldnewsapi.Configuration(
    host = "https://api.worldnewsapi.com"
)

# The client must configure the authentication and authorization parameters
# in accordance with the API server security policy.
# Examples for each auth method are provided below, use the example that
# satisfies your auth use case.

# Configure API key authorization: apiKey
configuration.api_key['apiKey'] = "0d033f49793b4de5a06b3cf353d2ed80"

# Uncomment below to setup prefix (e.g. Bearer) for API key, if needed
# configuration.api_key_prefix['apiKey'] = 'Bearer'

# Configure API key authorization: headerApiKey
configuration.api_key['headerApiKey'] = "0d033f49793b4de5a06b3cf353d2ed80"

# Uncomment below to setup prefix (e.g. Bearer) for API key, if needed
# configuration.api_key_prefix['headerApiKey'] = 'Bearer'

# Enter a context with an instance of the API client
with worldnewsapi.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = worldnewsapi.NewsApi(api_client)
    text = 'Suruhanjaya Syarikat Malaysia' # str | The text to match in the news content (at least 3 characters, maximum 100 characters). By default all query terms are expected, you can use an uppercase OR to search for any terms, e.g. tesla OR ford. You can also exclude terms by putting a minus sign (-) in front of the term, e.g. tesla -ford. For exact matches just put your term in quotes, e.g. \"elon musk\". (optional)
    text_match_indexes = 'title,content' # str | If a \"text\" is given to search for, you can specify where this text is searched for. Possible values are title, content, or both separated by a comma. By default, both title and content are searched. (optional)
    source_country = 'my' # str | The ISO 3166 country code from which the news should originate. (optional)
    # language = 'en' # str | The ISO 6391 language code of the news. (optional)
    # min_sentiment = -0.8 # float | The minimal sentiment of the news in range [-1,1]. (optional)
    # max_sentiment = 0.8 # float | The maximal sentiment of the news in range [-1,1]. (optional)
    earliest_publish_date = '2025-10-05 16:12:35' # str | The news must have been published after this date. (optional)
    latest_publish_date = '2025-10-14 16:12:35' # str | The news must have been published before this date. (optional)
    # news_sources = 'https://www.bbc.co.uk' # str | A comma-separated list of news sources from which the news should originate. (optional)
    # authors = 'John Doe' # str | A comma-separated list of author names. Only news from any of the given authors will be returned. (optional)
    # categories = 'politics,sports' # str | A comma-separated list of categories. Only news from any of the given categories will be returned. Possible categories are politics, sports, business, technology, entertainment, health, science, lifestyle, travel, culture, education, environment, other. Please note that the filter might leave out news, especially in non-English languages. If too few results are returned, use the text parameter instead. (optional)
    entities = 'ORG:Companies Commission of Malaysia' # str | Filter news by entities (see semantic types). (optional)
    # location_filter = '51.050407, 13.737262, 20' # str | Filter news by radius around a certain location. Format is \"latitude,longitude,radius in kilometers\". Radius must be between 1 and 100 kilometers. (optional)
    # sort = 'publish-time' # str | The sorting criteria (publish-time). (optional)
    # sort_direction = 'ASC' # str | Whether to sort ascending or descending (ASC or DESC). (optional)
    offset = 0 # int | The number of news to skip in range [0,100000] (optional)
    number = 10 # int | The number of news to return in range [1,100] (optional)

    try:
        # Search News
        api_response = api_instance.search_news(text=text, text_match_indexes=text_match_indexes, source_country=source_country, earliest_publish_date=earliest_publish_date, latest_publish_date=latest_publish_date, offset=offset, number=number)
        print("The response of NewsApi->search_news:\n")
        pprint(api_response)
    except Exception as e:
        print("Exception when calling NewsApi->search_news: %s\n" % e)

The response of NewsApi->search_news:

SearchNews200Response(offset=0, number=10, available=27, news=[SearchNews200ResponseNewsInner(summary='KUALA LUMPUR: Mahkamah Sesyen di sini, hari ini, diberitahu bahawa Sayed Amir Muzzakkir Al Sayed Mohamad menandatangani surat iringan untuk menyokong syarikat Nexuscorp Group Sdn Bhd dianugerahkan kontrak membekalkan Perkhidmatan Penyelenggaraan dan Pembekalan Alat Ganti bagi Peralatan Infrastruktur, Server, Perisian dan Radio Pengguna Sistem RMPNET untuk Polis Diraja Malaysia (PDRM).', image='https://assets.bharian.com.my/images/articles/BH7TANDATAN-O_BHfield_image_listing_featured.var_1759846193.jpg', sentiment=1.0, language='en', video=None, title='Bekas Setiausaha Politik Hamzah tandatangani surat iringan sokong Nexuscorp Group dianugerahkan kontrak - Saksi', url='https://www.bharian.com.my/berita/nasional/2025/10/1455513/bekas-setiausaha-politik-hamzah-tandatangani-surat-iringan-sokong', source_country='my', id=367044174, text='KUALA LUMPUR:

In [33]:
all_news_articles.extend(result.get('news'))

In [34]:
merged_news_output = {
                "total_available": len(all_news_articles),
                "news": all_news_articles
            }

In [38]:
merged_news_output.get('news')[0].get('publish_date')

'2025-10-22 04:28:15'